# CineMovie — Recommendation Model

## 1. TF-IDF Vectorization

In [1]:
# Import Pandas for working with our movie dataframe.
import pandas as pd

# Import TF-IDF Vectorizer to convert movie tags into numerical vectors.
from sklearn.feature_extraction.text import TfidfVectorizer

# Import cosine_similarity to measure similarity between movies.
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
# Load the cleaned movie dataset created during EDA.
new_movie_df = pd.read_csv('cleaned_movies.csv')

In [14]:
new_movie_df

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."
...,...,...,...
4804,9367,El Mariachi,El Mariachi just wants to play his guitar and ...
4805,72766,Newlyweds,A newlywed couple's honeymoon is upended by th...
4806,231617,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic..."
4807,126186,Shanghai Calling,When ambitious New York attorney Sam is sent t...


In [9]:
new_movie_df.shape

(4809, 3)

In [10]:
new_movie_df.dtypes

movie_id     int64
title       object
tags        object
dtype: object

## 2. TF-IDF Vectorization

In [11]:
# Create a TF-IDF vectorizer for converting movie tags into numerical features.
tfidf = TfidfVectorizer(max_features=5000,stop_words='english')

In [12]:
# TRANSFORM THE MOVIE TAGS
# Learn the vocabulary from the movie tags and transform each movie into a TF-IDF vector.
vectors = tfidf.fit_transform(new_movie_df['tags'])

In [13]:
# Display the shape of the TF-IDF matrix.
vectors.shape

(4809, 5000)

In [19]:
# Display the first 100 terms learned by the TF-IDF vectorizer.
tfidf.get_feature_names_out()[0:100]

array(['000', '007', '10', '100', '11', '12', '13', '14', '15', '16',
       '17', '18', '18th', '19', '1930s', '1940s', '1950s', '1960s',
       '1970s', '1980', '1980s', '1985', '1990s', '1999', '19th',
       '19thcentury', '20', '200', '2003', '2009', '20th', '21st', '23',
       '24', '25', '30', '300', '3d', '40', '50', '500', '60', '60s',
       '70', '70s', 'aaron', 'aaroneckhart', 'abandoned', 'abducted',
       'abigailbreslin', 'abilities', 'ability', 'able', 'aboard',
       'abuse', 'abusive', 'academic', 'academy', 'accept', 'accepted',
       'accepts', 'access', 'accident', 'accidental', 'accidentally',
       'accompanied', 'accomplish', 'account', 'accountant', 'accused',
       'ace', 'achieve', 'act', 'acting', 'action', 'actionhero',
       'actions', 'activist', 'activities', 'activity', 'actor', 'actors',
       'actress', 'acts', 'actual', 'actually', 'adam', 'adams',
       'adamsandler', 'adamshankman', 'adaptation', 'adapted', 'addict',
       'addicted', 'ad

In [20]:
# Display the total number of features learned by the vectorizer.
len(tfidf.get_feature_names_out())

5000

## 3. COSINE SIMILARITY

In [21]:
# Calculate the cosine similarity between every pair of movies.
similarity = cosine_similarity(vectors)

In [23]:
similarity

array([[1.        , 0.02190774, 0.01218387, ..., 0.00540869, 0.00609943,
        0.        ],
       [0.02190774, 1.        , 0.01246082, ..., 0.01728524, 0.        ,
        0.        ],
       [0.01218387, 0.01246082, 1.        , ..., 0.01623233, 0.        ,
        0.        ],
       ...,
       [0.00540869, 0.01728524, 0.01623233, ..., 1.        , 0.02911913,
        0.03276304],
       [0.00609943, 0.        , 0.        , ..., 0.02911913, 1.        ,
        0.01711901],
       [0.        , 0.        , 0.        , ..., 0.03276304, 0.01711901,
        1.        ]], shape=(4809, 4809))

In [24]:
similarity.shape

(4809, 4809)

##### Let's check with a single movie like avatar

In [25]:
# Find the index of Avatar in the movie dataframe.
movie_index = new_movie_df[new_movie_df['title'] == 'Avatar'].index[0]
movie_index

np.int64(0)

In [26]:
# Get the cosine similarity scores of Avatar with every movie.
avatar_similarity = similarity[movie_index]

In [27]:
# Check how many similarity scores we have.
len(avatar_similarity)

4809

In [29]:
# Sort movie indices according to their similarity with Avatar.
sorted_indices = sorted(enumerate(avatar_similarity), key=lambda x: x[1],reverse=True)

In [30]:
# Display the top 10 most similar movie indices and their similarity scores.
sorted_indices[:10]

[(0, np.float64(1.0)),
 (3729, np.float64(0.20357444133201313)),
 (582, np.float64(0.1973443580366979)),
 (3607, np.float64(0.18373365766286115)),
 (47, np.float64(0.17101253287806512)),
 (539, np.float64(0.1650459800039916)),
 (942, np.float64(0.16263847635970957)),
 (2405, np.float64(0.15881736196987076)),
 (1916, np.float64(0.15779102331480388)),
 (3537, np.float64(0.15314735085492293))]

In [31]:
# Display the titles corresponding to the most similar movie indices.
for index, score in sorted_indices[1:11]:
    print(new_movie_df.iloc[index]['title'], "→", round(score, 4))

Falcon Rising → 0.2036
Battle: Los Angeles → 0.1973
Apollo 18 → 0.1837
Star Trek Into Darkness → 0.171
Titan A.E. → 0.165
The Book of Life → 0.1626
Aliens → 0.1588
Lifeforce → 0.1578
Galaxina → 0.1531
Jarhead → 0.1514


## 4. Recommendation Function

In [54]:
# Create a function to return movies similar to a given movie.
def recommend(movie_title, top_n):

    # Check whether the requested movie exists in the dataset.
    if movie_title not in new_movie_df['title'].values:

        # Return a message when the movie is not found.
        return f"Movie '{movie_title}' was not found in the dataset."
    
    # Find the index of the movie entered by the user.
    movie_index = new_movie_df[new_movie_df['title'] == movie_title].index[0]

    # Get the similarity scores of the selected movie with every other movie.
    movie_similarity = similarity[movie_index]

    # Sort the movies according to their similarity scores in descending order.
    sorted_indices = sorted(enumerate(movie_similarity), key=lambda x: x[1], reverse=True)

    # Create an empty list to store the recommendations.
    recommendations = []

    # Select the top 10 similar movies, excluding the input movie itself.
    for index, score in sorted_indices[1:top_n+1]:

        # Get the title of the recommended movie.
        title = new_movie_df.iloc[index]['title']

        # Store the movie title and similarity score.
        recommendations.append({'title': title, 'similarity_score': round(score, 4)})

    # Convert the recommendations list into a DataFrame.
    return pd.DataFrame(recommendations)

In [55]:
# Test the updated recommendation function.
recommend('Avatar', top_n=5)

,title,similarity_score
0,Falcon Rising,0.2036
1,Battle: Los Angeles,0.1973
2,Apollo 18,0.1837
3,Star Trek Into Darkness,0.1710
4,Titan A.E.,0.1650


In [56]:
#Test the updated recommendation function.
recommend('Falcon Rising', top_n=5)

,title,similarity_score
0,Brother,0.2750
1,Showdown in Little Tokyo,0.2394
2,Battle: Los Angeles,0.2117
3,Avatar,0.2036
4,Jarhead,0.2012


In [57]:
recommend('Avtar', top_n=5)

"Movie 'Avtar' was not found in the dataset."

## 5. Testing the Recommendation System
**The recommendation engine is tested using representative movie titles
to verify that it produces relevant recommendations for valid inputs
and handles invalid inputs appropriately.**

##### Test 1 — Successful scenario: Avatar (SciFi)

In [67]:
# Test the recommendation system with a valid movie title.
avatar_recommendations = recommend('Avatar', top_n=5)

# Display the recommendations.
avatar_recommendations

,title,similarity_score
0,Falcon Rising,0.2036
1,Battle: Los Angeles,0.1973
2,Apollo 18,0.1837
3,Star Trek Into Darkness,0.1710
4,Titan A.E.,0.1650


##### Test 2 — Successful scenario: The Dark Knight Rises

In [68]:
# Test the recommendation system with another valid movie title.
dark_knight_recommendations = recommend('The Dark Knight Rises',top_n=5)

# Display the recommendations.
dark_knight_recommendations

,title,similarity_score
0,The Dark Knight,0.4560
1,Batman Returns,0.4002
2,Batman Begins,0.3536
3,Batman Forever,0.3367
4,Batman,0.3312


##### Test 3 — Successful scenario: Titanic (romantic/drama)

In [69]:
# Test the recommendation system with Titanic.
titanic_recommendations = recommend('Titanic', top_n=5)

# Display the recommendations.
titanic_recommendations

,title,similarity_score
0,Ghost Ship,0.2173
1,Poseidon,0.2070
2,In the Heart of the Sea,0.1954
3,Triangle,0.1889
4,Pirates of the Caribbean: On Stranger Tides,0.1858


##### Test 4 — Successful scenario: Toy Story (animated/family drama)

In [70]:
# Test the recommendation system with Toy Story.
toy_story_recommendations = recommend('Toy Story', top_n=5)

# Display the recommendations.
toy_story_recommendations

,title,similarity_score
0,Toy Story 3,0.5137
1,Toy Story 2,0.4947
2,The 40 Year Old Virgin,0.3446
3,Class of 1984,0.1784
4,Factory Girl,0.1758


##### Test 5 — Failure scenario: Invalid title

In [66]:
# Test how the system handles a movie title that does not exist.
invalid_recommendation = recommend('Avtar', top_n=5)

# Display the result.
invalid_recommendation

"Movie 'Avtar' was not found in the dataset."

##### Test 6 — Failure scenario: Empty input

In [72]:
# Test how the system handles an empty movie title.
empty_recommendation = recommend('',top_n = 5)

# Display the result.
empty_recommendation

"Movie '' was not found in the dataset."

In [74]:
# Test the recommendation system with Inception.
inception_recommendations = recommend('Inception', top_n=5)

# Display the recommendations.
inception_recommendations

,title,similarity_score
0,Don Jon,0.1728
1,Premium Rush,0.1504
2,Cypher,0.1452
3,Hesher,0.1353
4,Duplex,0.1308
